# Course Project - AIGC DETECTION Model training

This notebook is used to initialize data and train the optimal model.

## Section 1: Preprocess image
The image dataset provided by the problem shows that the AI-generated images are all 512x512x3 in size, while the sizes of the graphics captured by humans vary greatly. This will cause significant trouble for the subsequent model training. Therefore, it is necessary to unify the sizes of both to a fixed 512x512, and on this basis, retain the classification information and original file structure of the training set.


In [1]:
import os
from PIL import Image
import shutil
from pathlib import Path
import sys

In [ ]:
#Path configuration
SOURCE_DATASET = r"C:\Users\13426\AIGC-Detection-Dataset-2025"  #The path of the original dataset
TARGET_DATASET = r"C:\Users\13426\datasetai"  # The path for saving the preprocessed data
TARGET_SIZE = 512  # Uniform target size


In [2]:
def resize_with_padding(img, target_size, fill_color=(0, 0, 0)):
    """
    Core preprocessing function: Scale proportionally and fill into a square.
    """
    width, height = img.size
    scale = target_size / min(width, height)
    new_width = int(width * scale)
    new_height = int(height * scale)
    
    img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    new_img = Image.new('RGB', (target_size, target_size), fill_color)
    paste_x = (target_size - new_width) // 2
    paste_y = (target_size - new_height) // 2
    new_img.paste(img, (paste_x, paste_y))
    return new_img

In [3]:
def preprocess_dataset(source_root, target_root, target_size=512, extensions=('.jpg', '.jpeg', '.png', '.bmp')):
    """
    Preprocess the entire dataset.
    """
    source_root = Path(source_root)
    target_root = Path(target_root)
    
    # Traverse the source directory
    for split_dir in ['train', 'test']:  # Handle the training set and the test set
        split_source = source_root / split_dir
        split_target = target_root / split_dir
        
        if not split_source.exists():
            print(f"Warning: Source directory {split_source} doesn't exist. Skip it.")
            continue
        
        # For the training set, traverse the category folder; For the test set, process the images directly
        if split_dir == 'train':
            class_dirs = [d for d in split_source.iterdir() if d.is_dir()]
            for class_dir in class_dirs:
                class_name = class_dir.name
                process_folder(class_dir, split_target / class_name, target_size, extensions)
        else:
            process_folder(split_source, split_target, target_size, extensions, is_test=True)


In [4]:
def process_folder(source_folder, target_folder, target_size, extensions, is_test=False):
    """Process all the pictures within a single folder"""
    target_folder.mkdir(parents=True, exist_ok=True)
    
    image_files = []
    for ext in extensions:
        image_files.extend(source_folder.glob(f'*{ext}'))
        image_files.extend(source_folder.glob(f'*{ext.upper()}'))
    
    print(f"Handle folder [{source_folder}]，find {len(image_files)} pictures.")
    
    for i, img_path in enumerate(image_files):
        try:
            # Open and convert pictures
            with Image.open(img_path) as img:
                img = img.convert('RGB')
                processed_img = resize_with_padding(img, target_size)
            
            # build targer path
            if is_test:
                target_path = target_folder / img_path.name
            else:
                target_path = target_folder / img_path.name

            processed_img.save(target_path, 'JPEG', quality=95)
            
            if (i + 1) % 100 == 0:
                print(f" {i + 1}/{len(image_files)} pictures have been processed")
                
        except Exception as e:
            print(f"  An error occurred while processing the picture {img_path} : {e}")
    
    print(f"  Done! The picture has been saved to {target_folder}")


In [5]:
if __name__ == '__main__':
    print("Start preprocessing the dataset...")
    print(f"Source dictionary: {SOURCE_DATASET}")
    print(f"Target dictionary: {TARGET_DATASET}")
    print(f"Target size: {TARGET_SIZE}x{TARGET_SIZE}")
    print("-" * 50)
    
    # Perform preprocessing
    preprocess_dataset(SOURCE_DATASET, TARGET_DATASET, TARGET_SIZE)
    
    print("\n" + "=" * 50)
    print("The preprocessing of the dataset has been completed!")
    print(f"The preprocessed data has been saved to:{TARGET_DATASET}")
    print("You can now train using the new data path.")

开始预处理数据集...
源目录: C:\Users\13426\AIGC-Detection-Dataset-2025
目标目录: C:\Users\13426\datasetai
目标尺寸: 512x512
--------------------------------------------------
处理文件夹 [C:\Users\13426\AIGC-Detection-Dataset-2025\train\0_real]，找到 50000 张图片。
  已处理 100/50000 张图片
  已处理 200/50000 张图片
  已处理 300/50000 张图片
  已处理 400/50000 张图片
  已处理 500/50000 张图片
  已处理 600/50000 张图片
  已处理 700/50000 张图片
  已处理 800/50000 张图片
  已处理 900/50000 张图片
  已处理 1000/50000 张图片
  已处理 1100/50000 张图片
  已处理 1200/50000 张图片
  已处理 1300/50000 张图片
  已处理 1400/50000 张图片
  已处理 1500/50000 张图片
  已处理 1600/50000 张图片
  已处理 1700/50000 张图片
  已处理 1800/50000 张图片
  已处理 1900/50000 张图片
  已处理 2000/50000 张图片
  已处理 2100/50000 张图片
  已处理 2200/50000 张图片
  已处理 2300/50000 张图片
  已处理 2400/50000 张图片
  已处理 2500/50000 张图片
  已处理 2600/50000 张图片
  已处理 2700/50000 张图片
  已处理 2800/50000 张图片
  已处理 2900/50000 张图片
  已处理 3000/50000 张图片
  已处理 3100/50000 张图片
  已处理 3200/50000 张图片
  已处理 3300/50000 张图片
  已处理 3400/50000 张图片
  已处理 3500/50000 张图片
  已处理 3600/50000 张图片
  已处理 3700/50000 张图


![ipynb](preprocess_image.png)
From the figure, we have successfully unified the format of the pictures, facilitating the training of the model.

## Section 2: build data loader
This module defines how data is read, enhanced, and fed into the model, which is key to training efficiency. Create an efficient and configurable data pipeline (DataLoader), perform real-time data augmentation during training, and divide the training set, validation set, and test set.

Data Loader Architecture(As the picture)
Training set: ImageFolder automatic classification
Verification set: Randomly partitioned (20%)
Test set: Custom unlabeled loader
- Training set: ImageFolder automatic classification
- Verification set: Randomly partitioned (20%)
- Test set: Custom unlabeled loader

![ipynb](data_loader.png)

In [1]:
import os
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets
import numpy as np

In [2]:
class PaddingResize:
    """
    A serializable custom Transform class for scaling and filling images proportionally.
    """
    def __init__(self, target_size=512, fill_color=(0, 0, 0)):
        self.target_size = target_size
        self.fill_color = fill_color
    
    def __call__(self, img):
        """
        Scale the image proportionally and use fill_color with squares.
        """
        width, height = img.size
        scale = self.target_size / min(width, height)
        new_width = int(width * scale)
        new_height = int(height * scale)
        
        img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
        
        new_img = Image.new('RGB', (self.target_size, self.target_size), self.fill_color)
        paste_x = (self.target_size - new_width) // 2
        paste_y = (self.target_size - new_height) // 2
        new_img.paste(img, (paste_x, paste_y))
        
        return new_img

In [3]:
class UnlabeledTestDataset(Dataset):
    """
    A custom Dataset class for loading unlabeled test sets.
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = []
        
        # Traverse the folder and collect all supported image files
        valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
        for ext in valid_extensions:
            self.image_paths.extend(list(self.root_dir.glob(f'*{ext}')))
        
        # Sort by file name to ensure repeatability
        self.image_paths = sorted(self.image_paths)
        print(f"Find {len(self.image_paths)} images in test set.")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        
        try:
            # Open and convert the image to RGB
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"An error occurred while loading the picture {img_path} : {e}")
            # Return a placeholder image
            image = Image.new('RGB', (512, 512), (0, 0, 0))
        
        if self.transform:
            image = self.transform(image)
        
        return image, os.path.basename(img_path)

In [4]:
def get_transform_for_preprocessed(is_train=True, target_size=512):
    """
    Define the data transformation process for the preprocessed images. Add more data augmentation to prevent overfitting.
    """
    if is_train:
        # Training set: Stronger data augmentation
        transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.2),  # New addition: Random vertical flip
            transforms.RandomRotation(degrees=15),  # New addition: Random rotation
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),  
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  
            transforms.RandomResizedCrop(target_size, scale=(0.8, 1.0)), 
            transforms.RandomGrayscale(p=0.1),  # New addition: Random gray-scale transformation
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225]),
            transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),  
        ])
    else:
        # Validation set/test set: Only standardized and centrally trimmed
        transform = transforms.Compose([
            transforms.Resize(target_size + 32),  # Enlarge slightly
            transforms.CenterCrop(target_size),   # cut center
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
    return transform


In [5]:
def create_data_loaders(train_root, test_root, batch_size=32, num_workers=4, 
                        val_split=0.2, use_preprocessed=True, seed=42):
    """
    Create training, validation and test data loaders
    
    Parameter:
        train_root: Root directory of the train set
        test_root: Root directory of the test set
        batch_size: The number of pictures in each batch
        num_workers: The number of child processes used for data loading
        val_split: Proportion of validation set
        use_preprocessed: Whether to use the preprocessed image
        seed: Random seed
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    # Select the appropriate conversion process based on whether preprocessed data is used
    if use_preprocessed:
        train_transform = get_transform_for_preprocessed(is_train=True, target_size=512)
        val_transform = get_transform_for_preprocessed(is_train=False, target_size=512)
        test_transform = get_transform_for_preprocessed(is_train=False, target_size=512)
    else:
        train_transform = transforms.Compose([
            PaddingResize(target_size=512),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        val_transform = transforms.Compose([
            PaddingResize(target_size=512),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        test_transform = val_transform
    
    # Load the training set
    try:
        full_train_dataset = datasets.ImageFolder(root=train_root, transform=train_transform)
        print(f"Training set category: {full_train_dataset.classes}")
        print(f"Total number of training samples: {len(full_train_dataset)}")
        
        # Count the sample size by category
        class_counts = {}
        for class_idx in range(len(full_train_dataset.classes)):
            class_counts[full_train_dataset.classes[class_idx]] = sum(
                [1 for _, label in full_train_dataset.samples if label == class_idx]
            )
        print(f"Category distribution: {class_counts}")
        
        # Divide the training set and the validation set
        val_size = int(val_split * len(full_train_dataset))
        train_size = len(full_train_dataset) - val_size
        
        train_dataset, val_dataset = random_split(
            full_train_dataset, [train_size, val_size],
            generator=torch.Generator().manual_seed(seed)
        )
        
        print(f"The sample size of the training set: {len(train_dataset)}")
        print(f"The sample size of the veritificating set: {len(val_dataset)}")
        
    except Exception as e:
        print(f"Failed to load the training set {e}")
        print(f"Please check if the path is correct: {train_root}")
        raise
    
    # Test set: Use a custom unlabeled Dataset
    test_dataset = UnlabeledTestDataset(root_dir=test_root, transform=test_transform)
    
    # Build DataLoader
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if num_workers > 0 else False,
        drop_last=True  # Discard the last incomplete batch and keep the batch sizes consistent
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if num_workers > 0 else False,
        drop_last=False
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if num_workers > 0 else False,
        drop_last=False
    )
    
    print(f"Test set sample size: {len(test_dataset)}")
    
    return train_loader, val_loader, test_loader, full_train_dataset.classes


In [6]:
# Simple test code
if __name__ == '__main__':
    TRAIN_PATH = r"C:\Users\13426\datasetai\train"
    TEST_PATH = r"C:\Users\13426\datasetai\test"
    
    print("Test data loader...")
    try:
        train_loader, val_loader, test_loader, class_names = create_data_loaders(
            train_root=TRAIN_PATH,
            test_root=TEST_PATH,
            batch_size=16,
            num_workers=0,
            val_split=0.2,
            use_preprocessed=True
        )
        
        print("\nTest training set loading...")
        for images, labels in train_loader:
            print(f"Batch image shapes of the training set: {images.shape}, Label shape: {labels.shape}")
            print(f"Label distribution: {torch.bincount(labels)}")
            break
            
        print("\nTest verification set loading...")
        for images, labels in val_loader:
            print(f"Shape of validation set batch images: {images.shape}, Label shape {labels.shape}")
            break

        sample = next(iter(test_loader.dataset))
        print("Test set single sample format:", type(sample), len(sample))
        print("\nTest test set loading...")
        print(f"test_loader length: {len(test_loader)}") 

        for images, filenames in test_loader:
            print(f"Test set batch image shape: {images.shape}, File name example :{filenames[:3]}")
            break
            
        print(f"\nClass name: {class_names}")
        print("✅ The data loader has been created successfully!")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

Test data loader...
Training set category: ['0_real', '1_fake']
Total number of training samples: 50000
Category distribution: {'0_real': 25000, '1_fake': 25000}
The sample size of the training set: 40000
The sample size of the veritificating set: 10000
Find 2500 images in test set.
Test set sample size: 2500

Test training set loading...
Batch image shapes of the training set: torch.Size([16, 3, 512, 512]), Label shape: torch.Size([16])
Label distribution: tensor([10,  6])

Test verification set loading...
Shape of validation set batch images: torch.Size([16, 3, 512, 512]), Label shape torch.Size([16])
Test set single sample format: <class 'tuple'> 2

Test test set loading...
test_loader length: 157
Test set batch image shape: torch.Size([16, 3, 512, 512]), File name example :['image_0000001.jpg', 'image_0000002.jpg', 'image_0000003.jpg']

Class name: ['0_real', '1_fake']
✅ The data loader has been created successfully!


## Section 3: train model
We use three methods to optimise Resnet-50 model.

1：Parameter freezing strategy:
- Freeze layer1, layer2, conv1, bn1
- Verification set: Randomly partitioned (20%)
- Reduce trainable parameters by 60%+
  
2：Fully connected layer reconstruction:
- Adapt to binary classification tasks
- Dropout provides strong regularization
- Intermediate layer (256 dimensions) enhances feature combination capability
  
3：Five fold optimization strategy combination:
- Mixed precision: speed+memory efficiency
- Gradient Trimming: Training Stability
- AdamW: Better generalization
- Dynamic learning rate: Jumping out of local optima
- Dynamic learning rate: Jumping out of local optima

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import time
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc("font",family='YouYuan')
from torch.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split

In [29]:
#Parameter configuration 
TRAIN_ROOT = r"C:\Users\13426\datasetai\train"
TEST_ROOT = r"C:\Users\13426\datasetai\test"
    
TARGET_SIZE = 512
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_EPOCHS = 10  
LEARNING_RATE = 1e-4
    
# Regularization parameter
DROPOUT_RATE = 0.5
VAL_SPLIT_RATIO = 0.2  # Twenty percent of the training data is used as the validation set
PATIENCE = 5  # Early stop patience value
USE_AMP = True

In [30]:
def create_model(num_classes=2, device='cuda', dropout_rate=0.5):
    """
    Create and initialize the model, and add a Dropout layer to prevent overfitting.
    Use the ResNet-50 pre-trained on ImageNet and replace the final fully connected layer to adapt to the binary classification task.
    """
    # Load the pre-trained ResNet-50 model
    model = models.resnet50(weights='IMAGENET1K_V1')
    
    # Freeze some layers (train only the last few layers)
    for name, param in model.named_parameters():
        if 'layer1' in name or 'layer2' in name or 'conv1' in name or 'bn1' in name:
            param.requires_grad = False
    
    # Replace the final fully connected layer and add Dropout
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(dropout_rate/2),
        nn.Linear(256, num_classes)
    )
    
    model = model.to(device)
    
    # Print the number of trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}, Trainable parameters: {trainable_params:,}")
    
    return model

In [31]:
def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, scaler=None):
    """
    Train for one epoch.
    It integrates performance analysis code to distinguish data loading time from GPU computing time.
    Optional: Supports mixed-precision training (via scaler parameters).
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # --- Performance analysis variable  ---
    batch_load_time = 0.0    # The total loading time of batch data
    gpu_comp_time = 0.0      # Pure computing time of GPU
    analyzed_batches = 0
    
    epoch_start_time = time.time()
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        batch_start_time = time.time()
        
        images, labels = images.to(device, non_blocking=True), labels.to(device)
        
        data_transfer_end = time.time()
        batch_load_time += (data_transfer_end - batch_start_time)
        
        optimizer.zero_grad()
        
        # --GPU Computing Begins --
        gpu_start_time = time.time()
        
        if scaler is not None:

            with autocast(device_type=device.type):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:

            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        torch.cuda.synchronize()  # Ensure the accuracy of GPU timing
        gpu_end_time = time.time()
        # --GPU computing completed --
        
        gpu_comp_time += (gpu_end_time - gpu_start_time)
        analyzed_batches += 1
        
        # Calculation indicators
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Print performance analysis after processing a certain number of batches
        if (batch_idx + 1) % 50 == 0:
            avg_batch_load = batch_load_time / analyzed_batches
            avg_gpu_comp = gpu_comp_time / analyzed_batches
            gpu_utilization = (avg_gpu_comp / (avg_batch_load + avg_gpu_comp)) * 100 if (avg_batch_load + avg_gpu_comp) > 0 else 0
            
            print(f'[Performance analysis] Epoch {epoch:03d} | Batch {batch_idx+1:4d} | '
                  f'Loss: {loss.item():.4f} | '
                  f'GPU utilization: {gpu_utilization:.1f}%')
    
    epoch_time = time.time() - epoch_start_time
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    print(f'[Epoch {epoch:03d} Summary] Total time consumption: {epoch_time:.2f}s | '
          f'Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}%')
    
    return epoch_loss, epoch_acc


In [32]:
def validate(model, val_loader, criterion, device):
    """
    Verify on the validation set.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    val_loss = running_loss / len(val_loader)
    val_acc = 100. * correct / total
    return val_loss, val_acc

In [33]:
def predict_test_set(model, test_loader, device, class_names):
    """
    Predict the unlabeled test set and save the results to a CSV file.
    """
    model.eval()
    all_filenames = []
    all_predictions = []
    all_probabilities = []
    
    with torch.no_grad():
        for images, filenames in test_loader:
            images = images.to(device)
            outputs = model(images)
            
            # Obtain the prediction category and probability
            probabilities = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_filenames.extend(filenames)
            all_predictions.extend(predicted.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    results_df = pd.DataFrame({
        'filename': all_filenames,
        'prediction': all_predictions,
        'pred_label': [class_names[p] for p in all_predictions],
        'prob_fake': [prob[0] for prob in all_probabilities], 
        'prob_real': [prob[1] for prob in all_probabilities]  
    })
    
    return results_df


In [13]:
def main():
    # Set up the equipment
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    
    print(f"Training set path: {TRAIN_ROOT}")
    print(f"Test set path: {TEST_ROOT}")
    print(f"Batch size: {BATCH_SIZE}, Data loading process: {NUM_WORKERS}")
    print(f"Proportion of validation set: {VAL_SPLIT_RATIO}")
    print(f"Enable mixed-precision training: {USE_AMP}")
    
    # 1. Create a data loader (including the validation set)
    print("\n" + "="*50)
    print("Create a data loader...")
    train_loader, val_loader, test_loader, class_names = create_data_loaders(
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    val_split=VAL_SPLIT_RATIO,  # Make sure this parameter exists in data_loader.py
    use_preprocessed=True
)
    
    # 2. Initialize the model, loss function, and optimizer
    print("Initialize the model...")
    model = create_model(num_classes=2, device=device, dropout_rate=DROPOUT_RATE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    
    # Learning rate scheduler (using cosine annealing + hot restart)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=5, T_mult=2, eta_min=1e-6
    )
    
    # Gradient scaler with mixed precision training
    scaler = GradScaler() if (USE_AMP and str(device) == 'cuda') else None
    
    # 3. Training cycle
    print("\n" + "="*50)
    print("Begin training...")
    
    best_val_acc = 0.0
    patience_counter = 0
    train_history = {'loss': [], 'acc': [], 'val_loss': [], 'val_acc': []}
    
    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n--- Epoch {epoch}/{NUM_EPOCHS} ---")
        print(f"Current learning rate: {optimizer.param_groups[0]['lr']:.2e}")
        
        # Train a epoch
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, epoch, scaler
        )
        
        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        train_history['loss'].append(train_loss)
        train_history['acc'].append(train_acc)
        train_history['val_loss'].append(val_loss)
        train_history['val_acc'].append(val_acc)
        
        print(f'[Epoch {epoch:03d} result] Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | '
              f'Verificate Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%')
        
        scheduler.step()
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'train_acc': train_acc,
                'val_loss': val_loss,
                'val_acc': val_acc,
                'class_names': class_names
            }, 'best_model.pth')
            print(f'==> The best model has been saved to best_model.pth (Verify the accuracy rate: {val_acc:.2f}%)')
        else:
            patience_counter += 1
            print(f'The verification accuracy has not improved. Patience count: {patience_counter}/{PATIENCE}')
            
            if patience_counter >= PATIENCE:
                print(f"\nEarly stop trigger! Stop training at epoch {epoch}")
                break
    
    print("\n" + "="*50)
    print("Training complete！")
    print(f'Optimal verification accuracy: {best_val_acc:.2f}%')
    
if __name__ == '__main__':
    main()

Using device: cuda
Training set path: C:\Users\13426\datasetai\train
Test set path: C:\Users\13426\datasetai\test
Batch size: 32, Data loading process: 4
Proportion of validation set: 0.2
Enable mixed-precision training: True

Create a data loader...
训练集类别: ['0_real', '1_fake']
总训练样本数: 50000
类别分布: {'0_real': 25000, '1_fake': 25000}
训练集样本数: 40000
验证集样本数: 10000
在测试集中找到 2500 张图片。
测试集样本数: 2500
Initialize the model...
Total parameters: 24,033,090, Trainable parameters: 18,518,786

Begin training...

--- Epoch 1/10 ---
Current learning rate: 1.00e-04
[Performance analysis] Epoch 001 | Batch   50 | Loss: 0.1383 | GPU utilization: 97.4%
[Performance analysis] Epoch 001 | Batch  100 | Loss: 0.0298 | GPU utilization: 96.9%
[Performance analysis] Epoch 001 | Batch  150 | Loss: 0.0415 | GPU utilization: 96.8%
[Performance analysis] Epoch 001 | Batch  200 | Loss: 0.0203 | GPU utilization: 96.7%
[Performance analysis] Epoch 001 | Batch  250 | Loss: 0.0659 | GPU utilization: 96.6%
[Performance analys

## Section 4: Prediction, result analysis and visualization


Now that we have completed the training of the model, next we will make predictions on the test set, analyze the obtained prediction results, and create a visual image of the results to facilitate the comparison of the accuracy differences between the training set and the test set.

In [37]:
"""
Model evaluation script, used for analyzing model performance and checking overfitting
"""
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
import seaborn as sns
import pandas as pd
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [38]:
##Path configuration
model_path = 'best_model.pth'  
train_root=r"C:\Users\13426\datasetai\train"
test_root=r"C:\Users\13426\datasetai\test"

In [39]:
def evaluate_model(model_path, train_root, test_root):
    """
    Comprehensively evaluate the performance of the model
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"The equipment being used: {device}")
    
    # 1. Load model
    if not os.path.exists(model_path):
        print(f"Error: Model file {model_path} doesn't exist!")
        return None
    
    print(f"Loading model: {model_path}")
    checkpoint = torch.load(model_path, map_location=device)
    
    # 2. Create a data loader
    print("\nCreate a data loader...")
    try:
        # Use val_split=0.2 to obtain the validation set
        train_loader, val_loader, test_loader, class_names = create_data_loaders(
            train_root=train_root,
            test_root=test_root,
            batch_size=32,
            num_workers=4,
            val_split=0.2,  # Obtain the validation set
            use_preprocessed=True,
            seed=42  # Ensure repeatability
        )
        print(f"Category name: {class_names}")
        print(f"Training set batch: {len(train_loader)}")
        print(f"Validation set batch: {len(val_loader)}")
        print(f"Test set batch: {len(test_loader)}")
    except Exception as e:
        print(f"Failed to create the data loader: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # 3. Create and load the model
    print("\nInitialize the model...")
    try:
        model = create_model(num_classes=len(class_names), device=device, dropout_rate=0.0)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        print("The model has been loaded successfully!")
    except Exception as e:
        print(f"Model loading failed: {e}")
        import traceback
        traceback.print_exc()
        return None
    
    # 4. Check the training history
    print(f"\nModel checkpoint information:")
    for key in checkpoint.keys():
        if key not in ['model_state_dict', 'optimizer_state_dict', 'class_names']:
            print(f"  {key}: {checkpoint[key] if not isinstance(checkpoint[key], torch.Tensor) else checkpoint[key].item()}")
            
    '''
    To speed up the testing process, the evaluation test set and validation set are omitted. If necessary, comments can be removed
    # 5. Evaluate on the training set
    print("\n" + "="*50)
    print("Evaluate on the training set...")
    train_preds, train_labels, train_probs = get_predictions(model, train_loader, device)
    train_accuracy = np.mean(train_preds == train_labels) * 100
    print(f"Accuracy of the training set: {train_accuracy:.2f}% (Sample size: {len(train_labels)})")
    
    # 6. Evaluate on the validation set
    print("\n" + "="*50)
    print("Evaluate on the validation set...")
    val_preds, val_labels, val_probs = get_predictions(model, val_loader, device)
    val_accuracy = np.mean(val_preds == val_labels) * 100
    print(f"Verification set accuracy: {val_accuracy:.2f}% (Sample size: {len(val_labels)})")
    
    # 7. Print the detailed classification report
    print("\nTraining set classification report:")
    print(classification_report(train_labels, train_preds, 
                              target_names=class_names, digits=4))
    
    print("\nVerification set classification report:")
    print(classification_report(val_labels, val_preds, 
                              target_names=class_names, digits=4))
    
    # 8. Generate all visual charts
    print("\n" + "="*50)
    print("Generate all visual charts...")
    
    # 8.1 Draw the confusion matrix
    try:
        print("Generate confusion matrix...")
        plot_confusion_matrix(val_labels, val_preds, class_names, 
                            title='Validation set confusion matrix')
    except Exception as e:
        print(f"Failed to generate the confusion matrix: {e}")
        import traceback
        traceback.print_exc()
    
    # 8.2 Draw the ROC curve (only binary classification)
    try:
        print("Generate the ROC curve...")
        if len(class_names) == 2:
            plot_roc_curve(val_labels, val_probs[:, 1], class_names)
        else:
            print(f"ROC curve is only applicable to binary classification tasks，Currently here is {len(class_names)} classification")
    except Exception as e:
        print(f"The generation of the ROC curve failed: {e}")
        import traceback
        traceback.print_exc()
    
    # 8.3 Draw the precision-recall curve (only binary classification)
    try:
        print("Generate the precision-recall curve...")
        if len(class_names) == 2:
            plot_precision_recall_curve(val_labels, val_probs[:, 1], class_names)
        else:
            print(f"The PR curve is only applicable to binary classification tasks，currently here is{len(class_names)} classification")
    except Exception as e:
        print(f"Failed to generate the PR curve: {e}")
        import traceback
        traceback.print_exc()
    
    # 8.4 Draw a confidence distribution map
    try:
        print("Generate a confidence distribution map...")
        plot_confidence_distribution(train_probs, val_probs, class_names)
    except Exception as e:
        print(f"Failed to generate the confidence distribution map: {e}")
        import traceback
        traceback.print_exc()
    
    # 9. Overfitting analysis
    print("\n" + "="*50)
    print("Overfitting analysis:")
    print(f"Training accuracy: {train_accuracy:.2f}%")
    print(f"Verifing accuracy: {val_accuracy:.2f}%")
    accuracy_gap = abs(train_accuracy - val_accuracy)
    print(f"Accuracy gap: {accuracy_gap:.2f}%")
    
    if accuracy_gap > 10:
        print("⚠️  Warning: There may be severe overfitting!")
    elif accuracy_gap > 5:
        print("⚠️  Warning: There may be overfitting!")
    else:
        print("✅ The model has good generalization ability")
    '''
        
    # 10. Make predictions on the test set（No label）
    print("\n" + "="*50)
    print("Make predictions on the test set...")
    try:
        # The test set has no labels and is processed using dedicated functions
        test_predictions = predict_test_set_only(model, test_loader, device)
        
        # Save the prediction results of the test set
        save_test_predictions_only(test_predictions, class_names)
        
    except Exception as e:
        print(f"Test set prediction failed: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "="*50)
    print("The assessment is complete!")
    print("The generated file:")
    for file in ['confusion_matrix.png', 'roc_curve.png', 
                 'precision_recall_curve.png', 'confidence_distribution.png',
                 'test_predictions.csv']:
        if os.path.exists(file):
            print(f"  ✓ {file}")
        else:
            print(f"  ✗ {file} (not generated)")
    
    return model

In [40]:
def get_predictions(model, data_loader, device):
    """
    Obtain the model prediction results (applicable to labeled data)
    """
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    with torch.no_grad():
        for batch in data_loader:
            images, labels = batch 
            
            if isinstance(labels, list):
                labels = torch.tensor(labels)
            
            images = images.to(device)
            outputs = model(images)
            
            probabilities = torch.softmax(outputs, dim=1)
            _, predictions = torch.max(outputs, 1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    return np.array(all_predictions), np.array(all_labels), np.array(all_probabilities)

In [41]:
def predict_test_set_only(model, test_loader, device):
    """
    Only make predictions for the test set (without labels)
    """
    model.eval()
    all_predictions = []
    all_filenames = []
    all_probabilities = []
    
    with torch.no_grad():
        for batch in test_loader:
            # Test set return (images, filenames)
            images, filenames = batch
            
            images = images.to(device)
            outputs = model(images)
            
            # Obtain probabilities and predictions
            probabilities = torch.softmax(outputs, dim=1)
            _, predictions = torch.max(outputs, 1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_filenames.extend(filenames)
            all_probabilities.extend(probabilities.cpu().numpy())
    
    return {
        'filenames': all_filenames,
        'predictions': np.array(all_predictions),
        'probabilities': np.array(all_probabilities)
    }

In [42]:
def save_test_predictions_only(test_predictions, class_names):
    """
    Save the prediction results of the test set to CSV
    """

    cleaned_filenames = []
    for filename in test_predictions['filenames']:
        # Remove.jpg or other extensions
        if '.' in filename:
            cleaned_name = filename.rsplit('.', 1)[0]
        else:
            cleaned_name = filename
        cleaned_filenames.append(cleaned_name)
    '''
    results_df = pd.DataFrame({
        'ID': test_predictions['filename'],
        'label': test_predictions['prediction']
    })
    '''
    results_df = pd.DataFrame({
        'ID': cleaned_filenames,
        'label': test_predictions['predictions']
    })

    csv_path = 'test_predictions.csv'
    results_df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"The prediction results of the test set have been saved to: {csv_path}")

    return results_df


In [43]:
def plot_confusion_matrix(y_true, y_pred, class_names, title='混淆矩阵'):
    """
    Draw the confusion matrix
    """
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('"True label"')
    plt.xlabel('Predictive label')
    plt.tight_layout()
    
    save_path = 'confusion_matrix.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"The confusion matrix has been saved to: {save_path}")

In [44]:
def plot_roc_curve(y_true, y_scores, class_names):
    """
    Draw the ROC curve (only binary classification)
    """
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC曲线 (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random guess')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC curve')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    save_path = 'roc_curve.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"The ROC curve has been saved to: {save_path}, AUC: {roc_auc:.3f}")

In [45]:
def plot_precision_recall_curve(y_true, y_scores, class_names):
    """
    Draw the precision-recall curve (only binary classification)
    """
    precision, recall, _ = precision_recall_curve(y_true, y_scores)
    
    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, color='blue', lw=2, label='PR curve')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision - recall curve')
    plt.legend(loc="upper right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    save_path = 'precision_recall_curve.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"The precision-recall curve has been saved to: {save_path}")

In [46]:
def plot_confidence_distribution(train_probs, val_probs, class_names):
    """
    Draw the distribution of prediction confidence levels
    """
    plt.figure(figsize=(12, 5))
    
    # Draw the distribution of prediction confidence levels
    plt.subplot(1, 2, 1)
    train_confidences = train_probs.max(axis=1)
    plt.hist(train_confidences, bins=30, alpha=0.7, color='blue', edgecolor='black', label='训练集')
    plt.xlabel('Prediction confidence level')
    plt.ylabel('Sample size')
    plt.title('The confidence distribution of the training set prediction')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Add statistical information
    plt.text(0.05, 0.95, f'Mean: {train_confidences.mean():.3f}\nStandard deviation: {train_confidences.std():.3f}',
             transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # The confidence distribution of the validation set
    plt.subplot(1, 2, 2)
    val_confidences = val_probs.max(axis=1)
    plt.hist(val_confidences, bins=30, alpha=0.7, color='orange', edgecolor='black', label='验证集')
    plt.xlabel('Prediction confidence level')
    plt.ylabel('Sample size')
    plt.title('The confidence distribution of the validation set prediction')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Add statistical information
    plt.text(0.05, 0.95, f'Mean: {val_confidences.mean():.3f}\nStandard deviation: {val_confidences.std():.3f}',
             transform=plt.gca().transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    save_path = 'confidence_distribution.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"The confidence distribution map has been saved to:{save_path}")


In [47]:
if __name__ == '__main__':
    # Add command-line parameter support
    import argparse
    
    parser = argparse.ArgumentParser(description='Evaluate AI image detection models')
    parser.add_argument('--model', type=str, default=model_path , help='Model file path')
    parser.add_argument('--train_root', type=str, default=train_root, help='Root directory of the training set')
    parser.add_argument('--test_root', type=str, default=test_root, help='Root directory of the test set')
    
    args = parser.parse_args(args=[])
    
    evaluate_model(
        model_path=args.model,
        train_root=args.train_root,
        test_root=args.test_root
    )

The equipment being used: cuda
Loading model: best_model.pth

Create a data loader...
训练集类别: ['0_real', '1_fake']
总训练样本数: 50000
类别分布: {'0_real': 25000, '1_fake': 25000}
训练集样本数: 40000
验证集样本数: 10000
在测试集中找到 2500 张图片。
测试集样本数: 2500
Category name: ['0_real', '1_fake']
Training set batch: 1250
Validation set batch: 313
Test set batch: 79

Initialize the model...
Total parameters: 24,033,090, Trainable parameters: 18,518,786
The model has been loaded successfully!

Model checkpoint information:
  epoch: 5
  train_loss: 0.00660515604019165
  train_acc: 99.805
  val_loss: 0.00803544265679506
  val_acc: 99.79

Make predictions on the test set...
The prediction results of the test set have been saved to: test_predictions.csv

The assessment is complete!
The generated file:
  ✓ confusion_matrix.png
  ✓ roc_curve.png
  ✓ precision_recall_curve.png
  ✓ confidence_distribution.png
  ✓ test_predictions.csv


We trained the dataset using this model and obtained the loss rate and accuracy for each round. The model achieved the best results in the fourth round, with an accuracy of 99.8025% and a loss rate of 0.00699% for the training set. The accuracy of the validation set is 99.77%, with a loss rate of 0.00871%.


![ipynb](training_history.png)

Validate and evaluate the best trained model. Visualize the confidence distribution, confusion matrix, PR curve, and ROC curve of the validation set through visual means.

<img src="confusion_matrix.png" width="600" style="display: inline-block;"/>
<img src="precision_recall_curve.png" width="600" style="display: inline-block;"/>
<img src="roc_curve.png" width="600" style="display: inline-block;"/>
<img src="confidence_distribution.png" width="600" style="display: inline-block;"/>

Based on the validation and evaluation of the model, we can see that its prediction accuracy for this dataset is very high, exceeding 99%. From the comparison between the accuracy of training and validation, it can be seen that there is a 0.11% difference in accuracy between the two models regarding whether there is overfitting. We can consider that the model has good generalization ability and there is no overfitting. Finally, we applied the unclassified test set to obtain the final prediction results and saved them in a CSV file.


![ipynb](code_result.png)